**LangGraph is a useful Python library in the LangChain ecosystem. It helps build stateful, multi-actor applications like complex AI agents, multi-step workflows, and autonomous loops. With LangGraph, we can design AI workflows as graphs containing nodes for actions or functions and edges for transitions or paths, giving us full control over the execution flow.**

1. parallel execution in langraph
- parallel execution means multiple nodes run at then  same time 

In [1]:
from typing import TypedDict # 'typing' module se 'TypedDict' import kar rahe hain taaki hum state dictionary ka structure aur uska data type (string, int, etc.) strictly define kar sakein.
from langgraph.graph import StateGraph, END # 'langgraph.graph' package se 'StateGraph' (graph banane ke liye class) aur 'END' (graph execution ko khatam karne ka special marker) import kar rahe hain.

In [2]:
#state
class MyState(TypedDict): # ek custom state class bana rahe hain jo TypedDict ko inherit karti hai; yeh graph ke andar data ko ek node se doosre node tak pass karne ke schema ko define karti hai.
    topic:str
    python_result:str
    aws_result:str
    ml_result:str

#node 1
def pyhton_node(state):
    return{
        "pyhton_result": f"Python roadmap for {state['topic']} is variable,loop,function"

    }

#node 2
def aws_node(state):
    return{
        "aws_result": f"aws roadmap for {state['topic']} is ec2,s3, docker "

    }

#node 3
def ml_node(state):
    return{
        "ml_result": f"ml roadmap for {state['topic']} is Navie bayes, SVM,PCM "
    }

## built Graph
builder=StateGraph(MyState) # 'StateGraph' class ka ek object (builder) bana rahe hain aur ise 'MyState' schema pass kar rahe hain taaki yeh state ko track kar sake.
builder.add_node("python_node",pyhton_node)
builder.add_node("aws_node",aws_node)
builder.add_node("ml_node",ml_node)

##parallel execution from Start
builder.set_entry_point("python_node")
builder.set_entry_point("aws_node")
builder.set_entry_point("ml_node")

builder.add_edge("python_node", END)
builder.add_edge("aws_node",END)
builder.add_edge("ml_node",END)

##Compile Graph
app=builder.compile() # builder object ko compile karke ek executable runtime application (app) bana rahe hain jise hum invoke kar sakte hain.

##Run graph
result=app.invoke({
    "topic": "Data Engineering"
})

print(result)

  

{'topic': 'Data Engineering', 'aws_result': 'aws roadmap for Data Engineering is ec2,s3, docker ', 'ml_result': 'ml roadmap for Data Engineering is Navie bayes, SVM,PCM '}


2. Memory And CheckPointing
- Memory -> Memory means langgraph can remember previous conversation or previous state

  - ex.
   - user: My name is AAA
   - user : what is my name ?
   -  Graph remember: your name is AAA

- CheckPointing -> It is saving graph state after each step.
  - if graph stops in the middle,it can restart from saved state.

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import MemorySaver # 'MemorySaver' ek in-built checkpointing utility hai jo graph ki state ko RAM ya memory me save karti hai taaki purani baatcheet (conversation history) yaad rakhi ja sake.

In [4]:
#state
class chatState(TypedDict):
    user_name: str
    question: str
    answer: str
    

In [5]:
def chatbot_node(state):
    if "name" in state['question'].lower():
        return{
            "answer": f"Your name is {state['user_name']}"
        }
    else:
        return{
            "answer": "I do not understant your question"
        }

#build graph
builder= StateGraph(chatState)
builder.add_node("chatbot_node",chatbot_node)
builder.add_edge(START,"chatbot_node")
builder.add_edge("chatbot_node",END)

##add memory Saver
memory = MemorySaver()

app= builder.compile(checkpointer=memory) # graph ko compile karte waqt checkpointer argument me 'memory' pass kar rahe hain taaki state persist reh sake.
config={
    "configurable":{
        "thread_id": "user101" # ek unique thread_id de rahe hain taaki is specific user/session ki memory ko alag se track kiya ja sake.
    }
}

#First Call
result =app.invoke(
    {
        "user_name":"Krsn",
        "question": "what is my name?"
    },config=config # config pass kar rahe hain jisme thread_id hai taaki memory saver is session ko recognize kar sake.
)

print(result)


{'user_name': 'Krsn', 'question': 'what is my name?', 'answer': 'Your name is Krsn'}
